In [1]:
import sys 

sys.path.append("..")

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import torch.backends.cudnn as cudnn
import random
from mamba_ssm import Mamba

from thop import clever_format
from ptflops import get_model_complexity_info

import os
os.chdir("/workspace/dehazing")

In [2]:
def set_seed(seed):
    """Sets the seed for reproducibility across random, numpy, and torch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        cudnn.deterministic = True
        cudnn.benchmark = False

### Utility Code 

In [3]:
class LocalFeatureExtractor(nn.Module):
    """
    Refined for Mamba Block Integration.
    Focuses on edge-preservation and local consistency.
    """
    def __init__(self, dim, kernel_size=3, dilation=1):
        super().__init__()
        padding = (dilation * (kernel_size - 1)) // 2
        
        # We use a Depthwise-Pointwise structure to keep it fast
        self.conv = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size=kernel_size, padding=padding, 
                      dilation=dilation, groups=dim), # Local spatial context
            nn.BatchNorm2d(dim),
            nn.SiLU(),
            nn.Conv2d(dim, dim, kernel_size=1), # Inter-channel communication
            nn.BatchNorm2d(dim)
        )

    def forward(self, x):
        # We add the input back (Residual) so that even if the gate is 
        # closed, the original features aren't lost.
        return x + self.conv(x)

In [4]:
class PhysBiMambaBlock(nn.Module):
    """
    Bidirectional Mamba Block (BiMamba)
    Scans the image Forward AND Backward so the top-left pixel
    can 'see' the bottom-right pixel.|
    """
    def __init__(self, dim, dropout = 0.05):
        super().__init__()
        self.norm = nn.LayerNorm(dim)

        # Note: In true VMamba, they share the input projection layer to save memory. 
        # But keeping 4 separate Mambas is fine if you have the GPU RAM.
        self.mamba_h_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_h_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_v_fwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        self.mamba_v_bwd = Mamba(d_model=dim, d_state=16, d_conv=4, expand=2)
        
        # Fuses Fwd+Bwd direction
        self.fusion_proj = nn.Linear(dim, dim)

        # Smoothing to explicitly destroy 1D streaking artifacts
        self.spatial_smoothing = nn.Conv2d(dim, dim, kernel_size=3, padding=1, groups=dim)
        
        self.local_conv = LocalFeatureExtractor(dim, kernel_size=3, dilation=1)
        
        # Optional: A Gate to let the network choose emphasis
        self.mixer = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.Sigmoid()
        )
        
        # Initialize mixer bias negatively so local_conv is favored early in training
        nn.init.constant_(self.mixer[0].bias, -1.0)
        
        self.out_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, t_emb=None):
        B, C, H, W = x.shape
        residual = x
        
        x_flat = x.flatten(2).transpose(1, 2)
        x_norm = self.norm(x_flat)

        if t_emb is not None:
            scale, shift = t_emb.chunk(2, dim=1)
            x_norm = x_norm * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

        # ---------------------------------------------------------
        # 2. HORIZONTAL SCANS (Raster Order)
        # ---------------------------------------------------------
        # Forward ->
        out_h_fwd = self.mamba_h_fwd(x_norm)
        
        # Backward <-
        x_flip = torch.flip(x_norm, dims=[1])
        out_h_bwd = self.mamba_h_bwd(x_flip)
        out_h_bwd = torch.flip(out_h_bwd, dims=[1]) # Flip back

        # ---------------------------------------------------------
        # 3. VERTICAL SCANS (Column-Major Order)
        # ---------------------------------------------------------
        # Reshape to Image -> Transpose (Swap H and W) -> Flatten
        # Result: (B, W*H, C). Now 'neighbors' in seq are vertical neighbors.
        x_v_img = x_norm.view(B, H, W, C).permute(0, 2, 1, 3) 
        x_v_flat = x_v_img.flatten(1, 2)
        
        # Down v
        out_v_fwd = self.mamba_v_fwd(x_v_flat)
        
        # Up ^
        x_v_flip = torch.flip(x_v_flat, dims=[1])
        out_v_bwd = self.mamba_v_bwd(x_v_flip)
        out_v_bwd = torch.flip(out_v_bwd, dims=[1])
        
        # Un-Transpose Vertical Outputs back to Horizontal Order
        # (B, W*H, C) -> (B, W, H, C) -> (B, H, W, C) -> (B, L, C)
        out_v_fwd = out_v_fwd.view(B, W, H, C).permute(0, 2, 1, 3).flatten(1, 2)
        out_v_bwd = out_v_bwd.view(B, W, H, C).permute(0, 2, 1, 3).flatten(1, 2)
        
        ## ---------------------------------------------------------
        # 4. Global Fusion
        # ---------------------------------------------------------
        # Combine all 4 views of the image
        # WE are treating the outputs of the 4 directions seperate.
        # It mixes the channels of the 4 scans for a single pixel, but it does not communicate with neighboring pixels. 
        # The model simply overlays the horizontal streaks and vertical streaks on top of each other.
        # global_feat = self.fusion_linear(
        #     torch.cat([out_h_fwd, out_h_bwd, out_v_fwd, out_v_bwd], dim=-1)
        # )

        global_feat = out_h_fwd + out_h_bwd + out_v_fwd + out_v_bwd
        global_feat = self.fusion_proj(global_feat)

        # ---------------------------------------------------------
        # 5. NEW: Spatial Smoothing to remove streaks
        # ---------------------------------------------------------
        global_feat_img = global_feat.transpose(1, 2).view(B, C, H, W)
        global_feat_img = self.spatial_smoothing(global_feat_img)
        global_feat = global_feat_img.flatten(2).transpose(1, 2)

        # ---------------------------------------------------------
        # 6. Local Branch (Conv)
        # ---------------------------------------------------------
        # Reshape for Conv2d
        x_img_norm = x_norm.transpose(1, 2).view(B, C, H, W)
        local_feat = self.local_conv(x_img_norm)
        local_feat = local_feat.flatten(2).transpose(1, 2)

        
        # ---------------------------------------------------------
        # 6. Gated Output
        # ---------------------------------------------------------
        combined = torch.cat([global_feat, local_feat], dim=-1)
        z = self.mixer(combined)
        
        fused = global_feat * z + local_feat * (1 - z)
        
        x_out = self.out_proj(fused)
        
        # Reshape to (B, C, H, W) for residual add
        x_out = x_out.transpose(1, 2).view(B, C, H, W)
        x_out = self.dropout(x_out)
        
        return residual + x_out


### DehazeNet - Based Transmission

In [5]:
import torch
import torch.nn as nn

class BReLU(nn.Module):
    """
    Bilateral Rectified Linear Unit (BReLU)
    Bounds the output strictly between t_min and t_max (typically 0 and 1).
    """
    def __init__(self, t_min=0.0, t_max=1.0):
        super(BReLU, self).__init__()
        self.t_min = t_min
        self.t_max = t_max

    def forward(self, x):
        return torch.clamp(x, min=self.t_min, max=self.t_max)


class Maxout(nn.Module):
    """
    Maxout Activation Unit.
    Splits the channels into groups of size `num_pieces` and takes the maximum 
    across the channel dimension to reduce dimensionality.
    """
    def __init__(self, num_pieces):
        super(Maxout, self).__init__()
        self.num_pieces = num_pieces

    def forward(self, x):
        # x shape: (Batch, Channels, Height, Width)
        b, c, h, w = x.shape
        # Group channels and take the maximum
        x = x.view(b, c // self.num_pieces, self.num_pieces, h, w)
        return torch.max(x, dim=2)[0]


In [6]:
class DehazeNet_PhysMamba(nn.Module):
    def __init__(self, mamba_dim=16):
        super().__init__()

        # ---------------------------------------------------------
        # 1. Original DehazeNet Feature Extraction
        # ---------------------------------------------------------
        self.feature_extraction = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=5, padding=2),
            nn.SiLU() # Modernized from original BReLU for inner layers
        )

        # 2. Original DehazeNet Multi-Scale Mapping
        self.scale1 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
        self.scale2 = nn.Conv2d(16, 16, kernel_size=5, padding=2)
        self.scale3 = nn.Conv2d(16, 16, kernel_size=7, padding=3)

        # 3. Original Dehaze Local Extremum(Maxout)
        # 48 channels in -> maxout across 3 pieces -> 16 channels out
        self.local_extremum = Maxout(num_pieces=3)

        # ---------------------------------------------------------
        # 4. INJECT: Global Context via BiMamba
        # ---------------------------------------------------------
        # Takes the 16-channel structural features and correlates them globally
        self.global_context_block = PhysBiMambaBlock(dim=mamba_dim)


        # ---------------------------------------------------------
        # 5. Final Regression (t-map + A-vector)
        # ---------------------------------------------------------
        # Predicts transmission map
        self.t_regression = nn.Sequential(
            nn.Conv2d(mamba_dim, 1, kernel_size=3, padding=1),
            BReLU() # <--- CHANGED: Replaced Sigmoid with BReLU to keep edges sharp
        )

        # Predicts Atmospheric Light (A) from the global features for Flow Matching
        self.A_regression = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(mamba_dim, 3),
            BReLU() # <--- CHANGED: Replaced Sigmoid with BReLU to prevent saturated color estimates
        )

    def forward(self, x, t_emb = None):
        # Local structural extraction
        feat = self.feature_extraction(x)
        
        s1 = self.scale1(feat)
        s2 = self.scale2(feat)
        s3 = self.scale3(feat)
        multi_scale = torch.cat([s1, s2, s3], dim=1) # 48 channels
        
        local_feat = self.local_extremum(multi_scale) # 16 channels
        
        # Global consistency check (The BiMamba Magic)
        # Passes t_emb directly into the Mamba block for Flow Matching guidance
        global_feat = self.global_context_block(local_feat, t_emb=t_emb)
        
        # Final predictions
        t_map = self.t_regression(global_feat)
        A_vec = self.A_regression(global_feat)
        
        return t_map, A_vec

### Loss Functions

In [7]:
import torch.fft 
import os
import torch
import torch.nn.functional as F
import torchvision
import torch.fft
from torch.utils.tensorboard import SummaryWriter
from tqdm.notebook import tqdm
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure


def get_sobel_edges(img):
    """ Helper to extract edges using Sobel filters """
    kernel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], device=img.device).float().unsqueeze(0).unsqueeze(0)
    kernel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], device=img.device).float().unsqueeze(0).unsqueeze(0)
    
    # Repeat for channels if necessary, but transmission is 1-ch
    edges_x = F.conv2d(img, kernel_x, padding=1)
    edges_y = F.conv2d(img, kernel_y, padding=1)
    return torch.sqrt(edges_x**2 + edges_y**2 + 1e-6)


def get_fft_spectrum(img):
    """ Helper to visualize the frequency spectrum """
    # Compute 2D FFT and shift low frequencies to center
    fft = torch.fft.fft2(img)
    fft_shift = torch.fft.fftshift(fft)

    # Magnitude in log scale for visualization
    magnitude = torch.log(torch.abs(fft_shift) + 1e-6)

    # Normalize to [0, 1] for TensorBoard
    mag_min, mag_max = magnitude.min(), magnitude.max()

    return (magnitude - mag_min) / (mag_max - mag_min + 1e-6)


In [8]:
class CharbonnierLoss(nn.Module):
    def __init__(self, eps=1e-3):
        super(CharbonnierLoss, self). __init__()
        self.eps = eps

    def forward(self, x, y):
        diff = x - y
        loss = torch.mean(torch.sqrt(diff * diff + self.eps * self.eps))
        return loss


In [9]:
### We need to keep the structure as accurate as possible
class Stage1_RESIDELoss_Unified(nn.Module):
    def __init__(self, w_t=1.0, w_recon=0.5, w_struct=1.5):
        super().__init__()
        self.charbonnier = CharbonnierLoss()
        self.w_t = w_t
        self.w_recon = w_recon
        self.w_struct = w_struct
        
        # Pre-define kernels to save compute
        self.register_buffer('sobel_x', 
                             torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]).float().view(1,1,3,3))
        self.register_buffer('sobel_y', 
                             torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]).float().view(1,1,3,3))
        self.register_buffer('laplacian', 
                             torch.tensor([[-1,-1,-1], [-1,8,-1], [-1,-1,-1]]).float().view(1,1,3,3))

    def get_combined_gradients(self, x):
        grad_x = F.conv2d(x, self.sobel_x, padding=1)
        grad_y = F.conv2d(x, self.sobel_y, padding=1)
        sobel = torch.sqrt(grad_x**2 + grad_y**2 + 1e-6)
        
        lap = F.conv2d(x, self.laplacian, padding=1)
        # We return both, but they will be summed into one loss
        return sobel, lap

    def forward(self, pred_t_map, pred_A, gt_t_map, clean_img_01, hazy_img_01):
        # 1. Pixel-wise Anchor
        loss_t = self.charbonnier(pred_t_map, gt_t_map)

        # 2. Unified Structural Loss (Combats Redundancy)
        # We compute gradients for both pred and gt
        p_sobel, p_lap = self.get_combined_gradients(pred_t_map)
        g_sobel, g_lap = self.get_combined_gradients(gt_t_map)
        
        # Summing them BEFORE applying weight prevents gradient conflict
        # Sobel gets weight 1.0, Laplacian gets 0.5 (as a fine-tuner)
        loss_struct = self.charbonnier(p_sobel, g_sobel) + 0.5 * self.charbonnier(p_lap, g_lap)

        # 3. Physics Constraint
        I_recon = clean_img_01 * pred_t_map + pred_A.view(-1, 3, 1, 1) * (1.0 - pred_t_map)
        loss_recon = self.charbonnier(I_recon, hazy_img_01)

        total_loss = (self.w_t * loss_t) + (self.w_struct * loss_struct) + (self.w_recon * loss_recon)

        return total_loss, {
            "Total": total_loss.item(),
            "Pixel": loss_t.item(), 
            "Structural": loss_struct.item(),
            "Recon": loss_recon.item()
        }

In [10]:
import torch
import torch.nn.functional as F
from torchmetrics.image import StructuralSimilarityIndexMeasure, PeakSignalNoiseRatio

def gmsd_loss(pred, gt, c=0.0026):
    """
    Optimized Gradient Magnitude Similarity Deviation (GMSD).
    Expects tensors of shape [B, C, H, W] normalized to [0, 1].
    """
    B, C, H, W = pred.shape
    
    # 1. Define Prewitt Filters (Standard for GMSD)
    # Using 1/3 as per original paper; repeated for each input channel
    kernel_x = torch.tensor([[[[1/3, 0, -1/3], [1/3, 0, -1/3], [1/3, 0, -1/3]]]], 
                            device=pred.device, dtype=pred.dtype)
    kernel_x = kernel_x.repeat(C, 1, 1, 1) # Support RGB or Grayscale
    kernel_y = kernel_x.transpose(2, 3)

    # 2. Compute Gradients via Depthwise Convolution (Faster)
    # groups=C ensures each channel is processed independently
    p_gx = F.conv2d(pred, kernel_x, padding=1, groups=C)
    p_gy = F.conv2d(pred, kernel_y, padding=1, groups=C)
    p_mag = torch.sqrt(p_gx**2 + p_gy**2 + 1e-12)

    g_gx = F.conv2d(gt, kernel_x, padding=1, groups=C)
    g_gy = F.conv2d(gt, kernel_y, padding=1, groups=C)
    g_mag = torch.sqrt(g_gx**2 + g_gy**2 + 1e-12)

    # 3. Calculate Gradient Magnitude Similarity (GMS)
    # GMS is 1.0 where gradients are identical, lower elsewhere
    num = 2 * p_mag * g_mag + c
    den = p_mag**2 + g_mag**2 + c
    gms = num / den

    # 4. Standard Deviation (The "D" in GMSD)
    # The original paper suggests pooling/averaging the GMS map first 
    # if it's high res, but standard SD on the whole map is common.
    # We calculate SD over (C, H, W) for each image in batch
    score = torch.std(gms, dim=[1, 2, 3]) 
    
    return score.mean()


def evaluate_transmission_quality(pred_t, gt_t):
    """
    pred_t, gt_t: Tensors of shape [B, 1, H, W] in range [0, 1]
    """
    # 1. Standard PSNR (Pixel Accuracy)
    psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(pred_t.device)
    psnr_val = psnr_metric(pred_t, gt_t)

    # 2. SSIM (Structural Accuracy - fixes the PSNR paradox)
    ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(pred_t.device)
    ssim_val = ssim_metric(pred_t, gt_t)

    # 3. GMSD (Gradient/Edge Accuracy - Custom PyTorch implementation)
    gmsd_val = custom_gmsd(pred_t, gt_t)

    # 4. Total Variation (Measures "Cleanliness" / lack of noise)
    def total_variation(img):
        diff_h = torch.abs(img[:, :, 1:, :] - img[:, :, :-1, :]).sum()
        diff_w = torch.abs(img[:, :, :, 1:] - img[:, :, :, :-1]).sum()
        return diff_h + diff_w

    tv_val = total_variation(pred_t)

    return {
        "PSNR (High is good)": psnr_val.item(),
        "SSIM (High is good)": ssim_val.item(),
        "GMSD (LOW is good)": gmsd_val.item(),
        "TV_Smoothness": tv_val.item()
    }

### Getting the dataset

In [11]:
from data.utils import get_haze_transforms, partition_dataset
from torch.utils.data import Subset
from losses import CharbonnierLoss
import torch
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

In [12]:
class RESIDE_Indoor(Dataset):
    def __init__(self, dataset_path, transform=None):
        self.root_dir = Path(dataset_path)
        self.metadata_csv = pd.read_csv(self.root_dir / "metadata.csv")

        self.transform = transform
        self.data = []
        
        for idx, row in self.metadata_csv.iterrows():
            clean_path = self.root_dir / row["clear_image_path"]
            hazy_paths_str = row["hazy_image_paths"]
            hazy_image_paths = [
                path.strip()
                for path in hazy_paths_str.strip("[]").replace("'", "").split(",")
            ]
            list_hazy_paths = [
                self.root_dir / hazy_path for hazy_path in hazy_image_paths
            ]
            
            for hazy_path in list_hazy_paths:
                # --- NEW LOGIC: Deduce the Transmission Map Path ---
                # Example: hazy_path.name is "1_1_0.90179.png"
                hazy_filename = hazy_path.name
                parts = hazy_filename.split('_')
                
                # Reconstruct trans filename: "1_1.png"
                if len(parts) >= 2:
                    trans_filename = f"{parts[0]}_{parts[1]}.png"
                else:
                    trans_filename = hazy_filename # Fallback just in case
                
                trans_path = self.root_dir / "trans" / trans_filename
                # ---------------------------------------------------

                data_item = {
                    "index": idx, 
                    "clean": clean_path, 
                    "hazy": hazy_path,
                    "trans": trans_path # Store the trans path
                }

                self.data.append(data_item)

    def __repr__(self):
        return "RESIDE Indoor"

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data_item = self.data[idx]
        clean_path = data_item["clean"]
        hazy_path = data_item["hazy"]
        trans_path = data_item["trans"]

        try:
            clean_img = Image.open(clean_path).convert("RGB")
            hazy_img = Image.open(hazy_path).convert("RGB")
            # Load transmission map as Grayscale ("L")
            trans_img = Image.open(trans_path).convert("L") 
        except FileNotFoundError:
            print(f"Error: Missing image file at {clean_path}, {hazy_path}, or {trans_path}. Skipping")
            return self.__getitem__((idx + 1) % len(self))

        if self.transform:
            # IMPORTANT WARNING: 
            # If your 'get_haze_transforms' function only expects 2 inputs, 
            # you must update it to accept and return 3 inputs!
            clean_img, hazy_img, trans_img = self.transform(clean_img, hazy_img, trans_img)
        else:
            # Fallback tensorization
            clean_img = (
                torch.as_tensor(np.array(clean_img)).permute(2, 0, 1).float() / 255.0
            )
            hazy_img = (
                torch.as_tensor(np.array(hazy_img)).permute(2, 0, 1).float() / 255.0
            )
            # Add channel dimension to grayscale image (H, W) -> (1, H, W)
            trans_img = (
                torch.as_tensor(np.array(trans_img)).unsqueeze(0).float() / 255.0
            )

        return hazy_img, clean_img, trans_img

In [13]:
def get_reside_indoor_transforms(resize_size: int = 256):
    """
    Returns both train and val transforms strictly tuned for RESIDE-INDOOR.
    Safely handles the optional 3rd 'trans' (transmission) map.
    """
    
    # 1. Base Formatting: Convert to tensor [0, 1] -> Normalize to [-1, 1]
    to_tensor_norm = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
    ])

    # 2. Geometric Sync: Applied equally to clean, hazy, and trans to preserve alignment
    geometric_sync = v2.Compose([
        v2.RandomCrop(resize_size, pad_if_needed=True),
        v2.RandomHorizontalFlip(p=0.5),
        v2.RandomVerticalFlip(p=0.5),
    ])

    # 3. Color Jitter: Tuned specifically for RESIDE-INDOOR lighting
    color_jitter = v2.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.01)

    # --- TRAIN TRANSFORM ---
    def train_transform(clean, hazy, trans=None):
        # A. Apply spatial changes synchronously
        if trans is not None:
            clean, hazy, trans = geometric_sync(clean, hazy, trans)
        else:
            clean, hazy = geometric_sync(clean, hazy)

        # B. Apply color distortion ONLY to the hazy image
        hazy = color_jitter(hazy)

        # C. Tensor & Normalization
        clean, hazy = to_tensor_norm(clean), to_tensor_norm(hazy)
        
        if trans is not None:
            # Trans stays [0, 1] for physics math, NO normalization!
            trans = v2.functional.to_dtype(v2.functional.to_image(trans), torch.float32, scale=True)
            return clean, hazy, trans
            
        return clean, hazy

    # --- VAL TRANSFORM ---
    def val_transform(clean, hazy, trans=None):
        # Validation just normalizes. NO cropping or flipping.
        clean, hazy = to_tensor_norm(clean), to_tensor_norm(hazy)
        
        if trans is not None:
            trans = v2.functional.to_dtype(v2.functional.to_image(trans), torch.float32, scale=True)
            return clean, hazy, trans
            
        return clean, hazy

    return train_transform, val_transform

In [14]:
set_seed(42)

resolution = 256
verbose = True
num_subset_samples = 500

train_transform, val_transform = get_reside_indoor_transforms(resize_size = resolution)

data_path = "dataset/indoor-training-set/"
dataset = RESIDE_Indoor(dataset_path=data_path)

indices = torch.randperm(len(dataset))[:num_subset_samples].tolist()
subset_dataset = Subset(dataset, indices)

train_dataset, val_dataset = partition_dataset(
    subset_dataset,
    train_transform, 
    val_transform,
    train_ratio = 0.8
)

print(f"Total Subset Size: {len(subset_dataset)}")
print(f"Training Set Size: {len(train_dataset)}")
print(f"Validation Set Size: {len(val_dataset)}")

Total Subset Size: 500
Training Set Size: 400
Validation Set Size: 100


Training time

In [15]:
def total_variation(img):
    """ Measures 'Cleanliness' / lack of noise """
    diff_h = torch.abs(img[:, :, 1:, :] - img[:, :, :-1, :]).sum(dim=[1,2,3])
    diff_w = torch.abs(img[:, :, :, 1:] - img[:, :, :, :-1]).sum(dim=[1,2,3])
    return (diff_h + diff_w).mean()
    

In [16]:
def train_stage1_dehazenet_mamba(train_loader, val_loader,
                                 num_epochs=30, lr=1e-3, accum_iter=4, 
                                 w_t=1.0, w_recon=0.5, w_struct=1.5):
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1. Setup Naming & Logging
    run_name = f"DehazeMamba_wt{w_t}_wr{w_recon}_ws{w_struct}"
    print(f"🚀 Starting Stage 1 Training: {run_name} | Effective BS: {train_loader.batch_size * accum_iter}")
    
    log_dir = os.path.join("runs", "Stage1_Final", run_name)
    writer = SummaryWriter(log_dir=log_dir)

    # 2. Initialize Stateful Metrics
    psnr_metric = PeakSignalNoiseRatio(data_range=1.0).to(device)
    ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

    # 3. Model & Optimization
    model = DehazeNet_PhysMamba(mamba_dim=16).to(device)
    criterion = Stage1_RESIDELoss_Unified(w_t=w_t, w_recon=w_recon, w_struct=w_struct).to(device)
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    global_step = 0

    for epoch in range(num_epochs):
        # ==========================================
        #               TRAINING PHASE
        # ==========================================
        model.train()
        train_iterator = tqdm(train_loader, desc=f"Train Ep {epoch+1}/{num_epochs}", leave=True)
        optimizer.zero_grad()
        
        for i, (hazy, clean, trans_gt) in enumerate(train_iterator):
            hazy, clean, trans_gt = hazy.to(device), clean.to(device), trans_gt.to(device)

            pred_t, pred_A = model(hazy)
            
            # Use the unified loss function
            loss, loss_dict = criterion(pred_t, pred_A, trans_gt, clean, hazy)

            # Gradient Accumulation
            (loss / accum_iter).backward()

            if (i + 1) % accum_iter == 0 or (i + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

            if global_step % 20 == 0:
                for k, v in loss_dict.items():
                    writer.add_scalar(f"Train_Loss/{k}", v, global_step)

            global_step += 1
            train_iterator.set_postfix(loss=f"{loss.item():.4f}")

        # Update Scheduler
        scheduler.step()
        writer.add_scalar("Hyperparams/Learning_Rate", scheduler.get_last_lr()[0], epoch)
        
        # ==========================================
        #              VALIDATION PHASE
        # ==========================================
        model.eval()
        gmsd_accum = 0.0
        tv_accum = 0.0
        recon_psnr_accum = 0.0

        psnr_metric.reset()
        ssim_metric.reset()

        val_iterator = tqdm(val_loader, desc="Validating", leave=False)
        with torch.no_grad():
            for v_hazy, v_clean, v_trans in val_iterator:
                v_hazy, v_clean, v_trans = v_hazy.to(device), v_clean.to(device), v_trans.to(device)
                
                p_t, p_A = model(v_hazy)
                p_t = torch.clamp(p_t, 0.0, 1.0)
                p_A = torch.clamp(p_A, 0.0, 1.0)

                # Physics Reconstruction Check
                I_recon = v_clean * p_t + p_A.view(-1, 3, 1, 1) * (1.0 - p_t)
                I_recon = torch.clamp(I_recon, 0.0, 1.0)
                recon_mse = F.mse_loss(I_recon, v_hazy)
                recon_psnr_accum += 10 * torch.log10(1.0 / (recon_mse + 1e-8)).item()

                # Update advanced structural metrics
                psnr_metric.update(p_t, v_trans)
                ssim_metric.update(p_t, v_trans)
                gmsd_accum += gmsd_loss(p_t, v_trans).item()
                tv_accum += total_variation(p_t).item()

            num_batches = len(val_loader)
            avg_psnr = psnr_metric.compute()
            avg_ssim = ssim_metric.compute()
            avg_gmsd = gmsd_accum / num_batches
            avg_tv = tv_accum / num_batches
            avg_recon_psnr = recon_psnr_accum / num_batches

            # Log Quantitative Metrics
            writer.add_scalar("Metrics/PSNR", avg_psnr, epoch)
            writer.add_scalar("Metrics/SSIM", avg_ssim, epoch)
            writer.add_scalar("Metrics/GMSD", avg_gmsd, epoch) # Lower is better
            writer.add_scalar("Metrics/TV_Smoothness", avg_tv, epoch) # Lower is better
            writer.add_scalar("Diagnostics/Recon_PSNR", avg_recon_psnr, epoch)

            # ==========================================
            #      QUALITATIVE DIAGNOSTIC 6-ROW GRID
            # ==========================================
            # Get a static batch for visual consistency
            sample_hazy, sample_clean, sample_trans = next(iter(val_loader))
            s_h = sample_hazy[:4].to(device)
            s_c = sample_clean[:4].to(device)
            s_t_gt = sample_trans[:4].to(device)

            s_t_pred, s_A_pred = model(s_h)
            s_t_pred = torch.clamp(s_t_pred, 0, 1)

            # 1. Base maps (Expand 1-channel t_maps to 3-channel for RGB grid)
            vis_gt_t = s_t_gt.repeat(1, 3, 1, 1)
            vis_pred_t = s_t_pred.repeat(1, 3, 1, 1)

            # 2. Error Edges (Where did the model blur the map?)
            abs_err = torch.abs(s_t_pred - s_t_gt)
            err_edges = get_sobel_edges(abs_err).repeat(1, 3, 1, 1)

            # 3. FFT Fingerprints (Did we capture the high frequencies?)
            fft_gt_vis = get_fft_spectrum(s_t_gt).repeat(1, 3, 1, 1)
            fft_pred_vis = get_fft_spectrum(s_t_pred).repeat(1, 3, 1, 1)
            
            # Stack all 6 rows (4 images per row)
            # Row 1: Original Hazy
            # Row 2: Target Map
            # Row 3: Predicted Map
            # Row 4: Edge Error
            # Row 5: Target FFT
            # Row 6: Predicted FFT
            full_vis_stack = torch.cat([
                s_h, vis_gt_t, vis_pred_t, err_edges, fft_gt_vis, fft_pred_vis
            ], dim=0)

            # Create the master grid (nrow=4 means 4 columns, which gives us exactly 6 rows)
            grid = torchvision.utils.make_grid(full_vis_stack, nrow=4, normalize=False)
            writer.add_image('Diagnostic_Grid/All_Metrics', grid, epoch)

            # Optional: Log the Physics Reconstruction just to ensure A_vec isn't breaking
            s_recon = s_c * s_t_pred + s_A_pred.view(-1, 3, 1, 1) * (1.0 - s_t_pred)
            recon_grid = torchvision.utils.make_grid(torch.cat([s_h, s_recon], dim=0), nrow=4, normalize=True)
            writer.add_image('Reconstruction_Check/Hazy_vs_Recon', recon_grid, epoch)

        # Save Checkpoints
        if epoch == num_epochs - 1 or epoch % 10 == 0:
            os.makedirs("checkpoints", exist_ok=True)
            torch.save(model.state_dict(), f"checkpoints/{run_name}_ep{epoch}.pth")

    writer.close()
    print(f"✅ Finished | PSNR: {avg_psnr:.2f} | SSIM: {avg_ssim:.4f} | GMSD: {avg_gmsd:.4f}")
    return model

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DehazeNet_PhysMamba(mamba_dim=16).to(device)

macs_raw, params_raw = get_model_complexity_info(
    model, (3, 256, 256),
    as_strings=True,
    print_per_layer_stat=False, 
    verbose=False
)
print("Model has")
print(f"PARAMS {params_raw}, GFLOPS {macs_raw}") 


Model has
PARAMS 37.91 k, GFLOPS 1.6 GMac


In [18]:
train_loader = DataLoader(train_dataset, batch_size = 8, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 8, shuffle = False)

mamba_stage1 = train_stage1_dehazenet_mamba(train_loader, val_loader,
                                 num_epochs=30, lr=1e-3, accum_iter=4, 
                                 w_t=1.0, w_recon=0.5, w_struct=1.5)

🚀 Starting Stage 1 Training: DehazeMamba_wt1.0_wr0.5_ws1.5 | Effective BS: 32


Train Ep 1/30:   0%|          | 0/50 [00:00<?, ?it/s]

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

Train Ep 2/30:   0%|          | 0/50 [00:00<?, ?it/s]

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

...

Train Ep 29/30:   0%|          | 0/50 [00:00<?, ?it/s]

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

Train Ep 30/30:   0%|          | 0/50 [00:00<?, ?it/s]

Validating:   0%|          | 0/13 [00:00<?, ?it/s]

✅ Finished | PSNR: 16.95 | SSIM: 0.9194 | GMSD: 0.0696


In [19]:
def load_and_benchmark_mamba_stage1(checkpoint_path, val_loader, log_base="runs/Benchmarks/Model_Checks"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Setup Naming
    v_name = "Stage1_Final"
    writer = SummaryWriter(log_dir=os.path.join(log_base, v_name))

    # 2. Initialize Model
    model = DehazeNet_PhysMamba(mamba_dim=16).to(device)
    state_dict = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()

    # 3. Get the EXACT SAME fixed batch (using a seed)
    g = torch.Generator()
    g.manual_seed(42) 
    fixed_loader = DataLoader(val_loader.dataset, batch_size=4, shuffle=True, generator=g)
    s_h, s_c, s_t_gt = next(iter(fixed_loader))
    
    s_h, s_c, s_t_gt = s_h.to(device), s_c.to(device), s_t_gt.to(device)

    # 4. Inference
    with torch.no_grad():
        # PhysMamba Stage 1 returns 2 outputs: Transmission (t) and Atmospheric Light (A)
        s_t_pred, s_A_pred = model(s_h)
        s_t_pred = torch.clamp(s_t_pred, 0.0, 1.0)
        
        # --- PHYSICAL RECONSTRUCTION ---
        # I_recon = J*t + A*(1-t) 
        # (Using Ground Truth J since Stage 1 only predicts physical parameters)
        A_reshaped = torch.clamp(s_A_pred.view(-1, 3, 1, 1), 0.0, 1.0)
        s_h_recon = s_c * s_t_pred + A_reshaped * (1.0 - s_t_pred)
        s_h_recon = torch.clamp(s_h_recon, 0.0, 1.0)

        # 5. CALCULATE REQUESTED METRICS (Recon, Trans, Edge PSNR)
        
        # A. Reconstruction PSNR (Hazy vs Re-synthesized Hazy)
        recon_mse = F.mse_loss(s_h_recon, s_h)
        recon_psnr = 10 * torch.log10(1.0 / (recon_mse + 1e-8))
        
        # B. Transmission PSNR (Predicted t vs Target t)
        trans_mse = F.mse_loss(s_t_pred, s_t_gt)
        trans_psnr = 10 * torch.log10(1.0 / (trans_mse + 1e-8))
        
        # C. Edge PSNR (Error ONLY on the sharp edges of the Transmission map)
        gt_edges = get_sobel_edges(s_t_gt)
        edge_mask = (gt_edges > 0.1).float() 
        edge_mse = F.mse_loss(s_t_pred * edge_mask, s_t_gt * edge_mask)
        edge_psnr = 10 * torch.log10(1.0 / (edge_mse + 1e-8))


    # 6. Build Diagnostic Rows (Normalized individually to fix "Darkness" issue)
    vis_gt_t = s_t_gt.repeat(1, 3, 1, 1)
    vis_pred_t = s_t_pred.repeat(1, 3, 1, 1)

    # Edge Error Local Normalization
    abs_err = torch.abs(s_t_pred - s_t_gt)
    err_edges = get_sobel_edges(abs_err).repeat(1, 3, 1, 1)
    err_edges = err_edges / (err_edges.max() + 1e-8)

    # FFT Spectrum Local Normalization
    fft_gt_vis = get_fft_spectrum(s_t_gt).repeat(1, 3, 1, 1)
    fft_pred_vis = get_fft_spectrum(s_t_pred).repeat(1, 3, 1, 1)
    fft_gt_vis = fft_gt_vis / (fft_gt_vis.max() + 1e-8)
    fft_pred_vis = fft_pred_vis / (fft_pred_vis.max() + 1e-8)

    # 7. Stack the 8-Row Forensic Grid
    full_vis_stack = torch.cat([
        s_h,           # Row 1: Original Hazy (I)
        s_c,           # Row 2: Clean GT (J) 
        vis_gt_t,      # Row 3: Target Transmission (t_gt)
        vis_pred_t,    # Row 4: Predicted Transmission (t_pred)
        err_edges,     # Row 5: Normalized Edge Error
        fft_gt_vis,    # Row 6: Normalized FFT Target
        fft_pred_vis,  # Row 7: Normalized FFT Model
        s_h_recon      # Row 8: Reconstructed Hazy (The Physics Check)
    ], dim=0)

    # Final Grid (normalize=False prevents global brightness crushing)
    grid = torchvision.utils.make_grid(full_vis_stack, nrow=4, normalize=False)
    
    # 8. Log Everything to TensorBoard
    writer.add_image(f'benchmark_check/{v_name}', grid, 0)
    writer.add_scalar("Benchmark/Reconstruction_PSNR", recon_psnr, 0)
    writer.add_scalar("Benchmark/Transmission_PSNR", trans_psnr, 0)
    writer.add_scalar("Benchmark/Edge_PSNR", edge_psnr, 0)
    writer.add_scalar("Benchmark/Mean_A_Value", s_A_pred.mean(), 0)
    
    # Console Output
    print(f"--- {v_name} BENCHMARK COMPLETE ---")
    print(f"Reconstructed PSNR : {recon_psnr.item():.2f} dB")
    print(f"Transmission PSNR  : {trans_psnr.item():.2f} dB")
    print(f"Edge PSNR          : {edge_psnr.item():.2f} dB")
    print(f"Mean A-light       : {s_A_pred.mean().item():.4f}")
    
    writer.close()

In [20]:
checkpoint_path = "checkpoints/DehazeMamba_wt1.0_wr0.5_ws1.5_ep29.pth"
load_and_benchmark_mamba_stage1(checkpoint_path, val_loader)

--- Stage1_Final BENCHMARK COMPLETE ---
Reconstructed PSNR : 22.95 dB
Transmission PSNR  : 16.48 dB
Edge PSNR          : 31.09 dB
Mean A-light       : 0.8423
